# Evolution of Seismic Hotspots in Nepal (1990–2026)
### A Spatiotemporal Clustering Approach
**MINspire Earthquake Project**

## Setup & Imports

In [ ]:
import os
import io
import shutil
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

# Paths
BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Environment ready.')

## Phase 1 & 2: Load, Inspect & Clean Dataset

In [ ]:
# Auto-download via kagglehub if CSV not found
def get_data_path(data_dir):
    if os.path.exists(data_dir):
        csvs = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
        if csvs:
            return os.path.join(data_dir, csvs[0])
    import kagglehub
    print('Downloading dataset via kagglehub...')
    dl = kagglehub.dataset_download('amansinghnp/nepal-earthquake-seismicity-dataset-1990-2026')
    csvs = [f for f in os.listdir(dl) if f.endswith('.csv')]
    os.makedirs(data_dir, exist_ok=True)
    dest = os.path.join(data_dir, csvs[0])
    shutil.copy(os.path.join(dl, csvs[0]), dest)
    return dest

data_path = get_data_path(DATA_DIR)
df_raw = pd.read_csv(data_path)
print(f'Loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
df_raw.head()

In [ ]:
print('--- Data Types ---')
print(df_raw.dtypes)
print('\n--- Missing Values ---')
print(df_raw.isnull().sum())
print('\n--- Duplicates ---')
print(df_raw.duplicated().sum())

In [ ]:
df = df_raw.drop_duplicates().copy()

# Smart column mapping
col_map = {}
for col in df.columns:
    cl = col.lower()
    if cl in ['time','date','datetime']:     col_map['Datetime'] = col
    elif cl in ['latitude','lat']:           col_map['Latitude'] = col
    elif cl in ['longitude','lon','lng']:    col_map['Longitude'] = col
    elif cl == 'depth':                      col_map['Depth'] = col
    elif cl in ['mag','magnitude']:          col_map['Magnitude'] = col
# Fallback substring
for col in df.columns:
    cl = col.lower()
    if 'Datetime' not in col_map and ('time' in cl or 'date' in cl): col_map['Datetime'] = col
    elif 'Latitude' not in col_map and 'lat' in cl:                  col_map['Latitude'] = col
    elif 'Longitude' not in col_map and 'lon' in cl:                 col_map['Longitude'] = col
    elif 'Depth' not in col_map and 'depth' in cl:                   col_map['Depth'] = col
    elif 'Magnitude' not in col_map and 'mag' in cl and not any(x in cl for x in ['type','nst','source','error']): col_map['Magnitude'] = col

df = df.rename(columns={v: k for k, v in col_map.items()})

# Parse & coerce
df['Datetime'] = pd.to_datetime(df['Datetime'], errors='coerce')
for c in ['Latitude','Longitude','Depth','Magnitude']:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=['Datetime','Latitude','Longitude','Magnitude'])
if 'Depth' in df.columns: df['Depth'] = df['Depth'].fillna(df['Depth'].median())

print(f'Clean dataset: {df.shape}')
df[['Datetime','Latitude','Longitude','Depth','Magnitude']].describe()

## Phase 3: Feature Engineering

In [ ]:
df['Year']   = df['Datetime'].dt.year
df['Month']  = df['Datetime'].dt.month
df['Day']    = df['Datetime'].dt.day
df['Hour']   = df['Datetime'].dt.hour
df['Decade'] = (df['Year'] // 10) * 10

def season(m):
    if m in [12,1,2]:  return 'Winter'
    elif m in [3,4,5]: return 'Pre-monsoon'
    elif m in [6,7,8,9]: return 'Monsoon'
    else: return 'Post-monsoon'

df['Season'] = df['Month'].apply(season)
df['Mag_Category'] = pd.cut(df['Magnitude'], bins=[0,4,5,10], labels=['Minor','Moderate','Strong'])
df['Depth_Category'] = pd.cut(df['Depth'], bins=[0,70,300,10000], labels=['Shallow','Intermediate','Deep'])

def get_period(y):
    if y <= 1999: return '1990–1999'
    elif y <= 2009: return '2000–2009'
    elif y <= 2019: return '2010–2019'
    else: return '2020–2026'
df['Period'] = df['Year'].apply(get_period)

print('Features added:')
print(df[['Year','Month','Decade','Season','Mag_Category','Depth_Category','Period']].head(10))

## Phase 4: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Magnitude'], bins=30, kde=True, color='crimson', ax=axes[0])
axes[0].set_title('Magnitude Distribution')
axes[0].set_xlabel('Magnitude')
sns.boxplot(x=df['Magnitude'], color='orange', ax=axes[1])
axes[1].set_title('Magnitude Boxplot')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'magnitude_distribution.png'), dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Depth'], bins=30, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Depth Distribution')
axes[0].set_xlabel('Depth (km)')
sns.scatterplot(data=df, x='Depth', y='Magnitude', alpha=0.4, color='purple', ax=axes[1])
axes[1].set_title('Depth vs Magnitude')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'depth_analysis.png'), dpi=150)
plt.show()

In [ ]:
yearly = df['Year'].value_counts().sort_index()
plt.figure(figsize=(14, 5))
plt.plot(yearly.index, yearly.values, marker='o', color='darkblue', linewidth=2)
if 2015 in yearly.index:
    plt.axvline(x=2015, color='red', linestyle='--', label='2015 Gorkha Earthquake')
    plt.legend(fontsize=12)
plt.title('Earthquakes per Year (1990–2026)', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Count')
plt.savefig(os.path.join(OUTPUT_DIR, 'earthquakes_per_year.png'), dpi=150)
plt.show()
print(f'Peak year: {yearly.idxmax()} ({yearly.max()} events)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
month_counts = df['Month'].value_counts().sort_index()
axes[0].bar(month_counts.index, month_counts.values, color='teal')
axes[0].set_title('Earthquakes per Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Count')

season_order = ['Winter','Pre-monsoon','Monsoon','Post-monsoon']
season_counts = df['Season'].value_counts().reindex(season_order)
axes[1].bar(season_counts.index, season_counts.values, color='coral')
axes[1].set_title('Earthquakes by Season')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'temporal_analysis.png'), dpi=150)
plt.show()

## Phase 5: Spatial Analysis

In [ ]:
plt.figure(figsize=(12, 8))
sc = plt.scatter(
    df['Longitude'], df['Latitude'],
    c=df['Magnitude'], cmap='YlOrRd',
    s=df['Magnitude']**2 * 3, alpha=0.5, edgecolors='k', linewidths=0.2
)
plt.colorbar(sc, label='Magnitude')
plt.title('Geographic Distribution of Earthquake Epicenters', fontsize=14)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.savefig(os.path.join(OUTPUT_DIR, 'spatial_distribution.png'), dpi=150)
plt.show()

## Phase 6: Seismic Hotspot Detection (DBSCAN & K-Means)

In [ ]:
coords = df[['Latitude','Longitude']].values
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

# DBSCAN
dbscan = DBSCAN(eps=0.2, min_samples=10)
df['DBSCAN_Cluster'] = dbscan.fit_predict(coords_scaled)
n_clusters = len(set(df['DBSCAN_Cluster'])) - (1 if -1 in df['DBSCAN_Cluster'].values else 0)
n_noise = (df['DBSCAN_Cluster'] == -1).sum()
print(f'DBSCAN: {n_clusters} clusters found, {n_noise} noise points')

In [ ]:
plt.figure(figsize=(12, 8))
palette = sns.color_palette('tab10', n_colors=n_clusters+1)
for cid in sorted(df['DBSCAN_Cluster'].unique()):
    subset = df[df['DBSCAN_Cluster'] == cid]
    label = f'Noise' if cid == -1 else f'Cluster {cid}'
    color = 'lightgrey' if cid == -1 else palette[cid]
    plt.scatter(subset['Longitude'], subset['Latitude'], label=label, s=10, alpha=0.6, c=[color])
plt.legend(loc='lower right', fontsize=9, markerscale=2)
plt.title('DBSCAN Seismic Hotspots', fontsize=14)
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.savefig(os.path.join(OUTPUT_DIR, 'dbscan_hotspots.png'), dpi=150)
plt.show()

In [ ]:
# KMeans - Elbow method
inertias = []
k_range = range(2, 10)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km.fit(coords_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), inertias, marker='o', color='green')
plt.title('K-Means Elbow Method')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.savefig(os.path.join(OUTPUT_DIR, 'kmeans_elbow.png'), dpi=150)
plt.show()

In [ ]:
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
df['KMeans_Cluster'] = kmeans.fit_predict(coords_scaled)

plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='Longitude', y='Latitude', hue='KMeans_Cluster', palette='Set1', alpha=0.6, s=15)
plt.title(f'K-Means Clustering (K={optimal_k})', fontsize=14)
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.savefig(os.path.join(OUTPUT_DIR, 'kmeans_hotspots.png'), dpi=150)
plt.show()

## Phase 7: Hotspot Evolution Over Time

In [ ]:
periods = ['1990–1999','2000–2009','2010–2019','2020–2026']
fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=True, sharey=True)
axes = axes.flatten()

for idx, period in enumerate(periods):
    pdf = df[df['Period'] == period].copy()
    ax = axes[idx]
    if len(pdf) < 10:
        ax.set_title(f'{period} (insufficient data)')
        continue
    sc = StandardScaler()
    pc = sc.fit_transform(pdf[['Latitude','Longitude']].values)
    c = DBSCAN(eps=0.25, min_samples=5).fit_predict(pc)
    pdf['PCluster'] = c
    n_c = len(set(c)) - (1 if -1 in c else 0)
    for cid in sorted(pdf['PCluster'].unique()):
        sub = pdf[pdf['PCluster'] == cid]
        col = 'lightgrey' if cid == -1 else None
        ax.scatter(sub['Longitude'], sub['Latitude'], s=8, alpha=0.6,
                   color=col if col else plt.cm.tab10(cid / max(n_c, 1)))
    ax.set_title(f'{period}  |  Clusters: {n_c}  |  Events: {len(pdf)}')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

plt.suptitle('Seismic Hotspot Evolution by Period', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'hotspot_evolution.png'), dpi=150, bbox_inches='tight')
plt.show()

## Phase 8: Statistical Analysis

In [ ]:
decade_stats = df.groupby('Decade').agg(
    Events=('Magnitude','count'),
    Avg_Magnitude=('Magnitude','mean'),
    Max_Magnitude=('Magnitude','max'),
    Avg_Depth=('Depth','mean')
).round(2)
print('=== Decadal Statistics ===')
display(decade_stats)

In [ ]:
corr_cols = [c for c in ['Magnitude','Depth','Latitude','Longitude','Year'] if c in df.columns]
corr = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'correlation_matrix.png'), dpi=150)
plt.show()

## Phase 9: Risk Zone Identification

In [ ]:
cluster_df = df[df['DBSCAN_Cluster'] != -1]
risk = cluster_df.groupby('DBSCAN_Cluster').agg(
    Event_Count=('Magnitude','count'),
    Avg_Magnitude=('Magnitude','mean'),
    Max_Magnitude=('Magnitude','max'),
    Avg_Depth=('Depth','mean'),
    Avg_Lat=('Latitude','mean'),
    Avg_Lon=('Longitude','mean')
).reset_index().round(2)

q75 = risk['Event_Count'].quantile(0.75)
q50 = risk['Event_Count'].median()

def assign_risk(r):
    if r['Event_Count'] > q75 or r['Max_Magnitude'] >= 6.0: return 'High Risk'
    elif r['Event_Count'] > q50 or r['Avg_Magnitude'] >= 4.5: return 'Moderate Risk'
    return 'Low Risk'

risk['Risk_Level'] = risk.apply(assign_risk, axis=1)
risk.to_csv(os.path.join(OUTPUT_DIR, 'risk_zones_summary.csv'), index=False)
print('=== Risk Zone Summary ===')
display(risk)

In [ ]:
# Visualize risk zones on map
color_map = {'High Risk': 'red', 'Moderate Risk': 'orange', 'Low Risk': 'green'}
plt.figure(figsize=(12, 8))

noise = df[df['DBSCAN_Cluster'] == -1]
plt.scatter(noise['Longitude'], noise['Latitude'], c='lightgrey', s=5, alpha=0.3, label='Noise')

for _, row in risk.iterrows():
    cdata = df[df['DBSCAN_Cluster'] == row['DBSCAN_Cluster']]
    c = color_map[row['Risk_Level']]
    plt.scatter(cdata['Longitude'], cdata['Latitude'], c=c, s=15, alpha=0.6,
                label=f"Cluster {int(row['DBSCAN_Cluster'])} ({row['Risk_Level']})")

plt.title('Seismic Risk Zones (based on DBSCAN clusters)', fontsize=14)
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.legend(loc='lower right', fontsize=9)
plt.savefig(os.path.join(OUTPUT_DIR, 'risk_zones_map.png'), dpi=150)
plt.show()

## Phase 10: Conclusions

In [ ]:
print('=== PROJECT CONCLUSIONS ===')
print(f'Total earthquakes analyzed: {len(df)}')
print(f'Date range: {df["Datetime"].min().date()} to {df["Datetime"].max().date()}')
print(f'Magnitude range: {df["Magnitude"].min():.1f} – {df["Magnitude"].max():.1f}')
print(f'Max magnitude event: {df.loc[df["Magnitude"].idxmax(), "Magnitude"]} on {df.loc[df["Magnitude"].idxmax(), "Datetime"].date()}')
print()
print('Decadal Summary:')
print(decade_stats.to_string())
print()
print('Risk Zones:')
for _, row in risk.iterrows():
    print(f'  Cluster {int(row["DBSCAN_Cluster"])}: {row["Risk_Level"]} ({int(row["Event_Count"])} events, Max Mag {row["Max_Magnitude"]})')
print()
print('Key Findings:')
print('  1. Seismic activity has consistently concentrated in the central-eastern Nepal region.')
print('  2. The 2010s saw the highest event count, dominated by 2015 Gorkha aftershocks.')
print('  3. Average depths have decreased over decades (shallower), indicating near-surface fault reactivation.')
print('  4. DBSCAN effectively identifies irregular-shaped hotspots unlike K-Means.')
print('  5. Persistent high-risk zones exist regardless of the time period analyzed.')
print()
print(f'All outputs saved to: {OUTPUT_DIR}')